# The Bitcoin L2 Trust Spectrum
### Quantifying What You Actually Trust When You Use Lightning, Spark, and Liquid

**Author:** Josh Homan, Homan Quantitative Solutions LLC  
**Date:** May 2026

This notebook pulls live data from three free APIs (mempool.space, CoinGecko, Blockstream Liquid), builds a comparison matrix across Bitcoin scaling layers, and visualizes adoption and cost tradeoffs.

The thesis: most Bitcoin Layer 2 (L2) discussion focuses on Total Value Locked (TVL) and transaction throughput, but rarely on the trust assumptions users accept when moving from base-chain Bitcoin to off-chain rails. This notebook quantifies those tradeoffs.

## 1. Setup

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

# Helper that returns None on any API failure so one broken endpoint
# does not kill the whole notebook
def safe_get(url, timeout=15):
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"Request failed for {url[:60]}...: {e}")
        return None

## 2. Lightning Network Live Stats

Pulling from mempool.space's free public API. This returns current public-channel capacity, channel count, and node count. Note: this captures only public Lightning Network (LN) channels, so the true network is larger.

In [ ]:
ln_stats = safe_get("https://mempool.space/api/v1/lightning/statistics/latest")

if ln_stats and 'latest' in ln_stats:
    latest = ln_stats['latest']
    ln_capacity_btc = latest.get('total_capacity', 0) / 1e8  # sats to BTC
    ln_channels = latest.get('channel_count', 0)
    ln_nodes = latest.get('node_count', 0)
    print(f"Lightning Network public capacity: {ln_capacity_btc:,.2f} BTC")
    print(f"Public channels: {ln_channels:,}")
    print(f"Public nodes: {ln_nodes:,}")
else:
    # Fallback values if API is down, update manually from mempool.space
    ln_capacity_btc = 5600
    ln_channels = 48000
    ln_nodes = 12000
    print("Using fallback values, update from mempool.space if needed")

In [ ]:
# Pull historical Lightning capacity for the chart later
ln_history = safe_get("https://mempool.space/api/v1/lightning/statistics/3y")

if ln_history:
    ln_df = pd.DataFrame(ln_history)
    ln_df['date'] = pd.to_datetime(ln_df['added'], unit='s')
    ln_df['capacity_btc'] = ln_df['total_capacity'] / 1e8
    # Keep last 365 days
    ln_df = ln_df[ln_df['date'] >= (pd.Timestamp.now() - pd.Timedelta(days=365))]
    ln_df = ln_df.sort_values('date').reset_index(drop=True)
    print(f"Loaded {len(ln_df)} days of Lightning capacity data")
    ln_df.tail()
else:
    ln_df = None
    print("Lightning history unavailable, chart will skip this layer")

## 3. Bitcoin Price History

CoinGecko free tier, 365 days of daily Bitcoin (BTC) price.

In [ ]:
btc_data = safe_get(
    "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"
    "?vs_currency=usd&days=365&interval=daily"
)

if btc_data and 'prices' in btc_data:
    btc_df = pd.DataFrame(btc_data['prices'], columns=['timestamp', 'price'])
    btc_df['date'] = pd.to_datetime(btc_df['timestamp'], unit='ms')
    btc_df = btc_df[['date', 'price']]
    current_btc_price = btc_df['price'].iloc[-1]
    print(f"Current BTC price: ${current_btc_price:,.2f}")
    print(f"Loaded {len(btc_df)} days of price data")
    btc_df.tail()
else:
    btc_df = None
    current_btc_price = 100000  # fallback
    print("Using fallback BTC price")

## 4. Liquid Network L-BTC Supply

Blockstream Liquid API. The asset ID below is the canonical Liquid Bitcoin (L-BTC) asset. Issuance tracking gives us how much BTC is currently pegged into the Liquid sidechain.

In [ ]:
LBTC_ASSET_ID = "6f0279e9ed041c3d710a9f57d0c02928416460c4b722ae3457a11eec381c526d"
liquid_data = safe_get(f"https://blockstream.info/liquid/api/asset/{LBTC_ASSET_ID}")

if liquid_data:
    issued = liquid_data.get('chain_stats', {}).get('issued_amount', 0) / 1e8
    burned = liquid_data.get('chain_stats', {}).get('burned_amount', 0) / 1e8
    liquid_btc = issued - burned
    print(f"L-BTC currently in circulation: {liquid_btc:,.2f} BTC")
else:
    liquid_btc = 3000  # fallback, update from blockstream.info/liquid
    print(f"Using fallback Liquid value: {liquid_btc} BTC")

## 5. Spark Adoption (Manual)

Spark is too new for a clean public API. Update these values from Lightspark's public disclosures, blog posts, or X announcements as they publish them. Cite sources in your final report.

In [ ]:
# Update these from latest Lightspark disclosures
# Source: https://www.lightspark.com (check blog and X for latest figures)
spark_btc_estimated = 50  # placeholder, update with real number
spark_source_note = "Lightspark public disclosures as of [DATE], see report for citation"
print(f"Spark estimated BTC: ~{spark_btc_estimated} BTC ({spark_source_note})")

## 6. The Trust Matrix

This is the analytical centerpiece. Each row is a Bitcoin custody or transfer rail. Columns describe the trust assumptions, exit guarantees, and operational characteristics.

**Why this matters:** users routinely conflate "non-custodial" with "trustless" but the trust models across these rails differ substantially. Quantifying them side by side reveals tradeoffs that pure TVL comparisons miss.

In [ ]:
trust_matrix = pd.DataFrame([
    {
        'Layer': 'L1 Bitcoin',
        'Custody Model': 'Self-custody (single key or multisig)',
        'Trusted Parties': 0,
        'Exit Mechanism': 'Already on L1',
        'Time to Unilateral Exit': 'N/A',
        'Online Requirement': 'Sender only at send time',
        'Transfer Speed': '10 to 60 min',
        'Typical Fee Range': '$1 to $50',
    },
    {
        'Layer': 'Lightning Network',
        'Custody Model': 'Self-custody, channel-based 2-of-2',
        'Trusted Parties': 1,  # channel counterparty
        'Exit Mechanism': 'Force-close channel to L1',
        'Time to Unilateral Exit': '~1 day to 2 weeks (CSV timelock)',
        'Online Requirement': 'Recipient must be online',
        'Transfer Speed': '<1 second',
        'Typical Fee Range': '<$0.01',
    },
    {
        'Layer': 'Spark',
        'Custody Model': 'Threshold multisig, 1-of-n honest operator',
        'Trusted Parties': 5,  # estimate, varies by operator set
        'Exit Mechanism': 'Pre-signed L1 exit transaction',
        'Time to Unilateral Exit': 'Hours to days (timelock dependent)',
        'Online Requirement': 'Recipient can be offline',
        'Transfer Speed': '<1 second',
        'Typical Fee Range': 'Free off-chain',
    },
    {
        'Layer': 'Liquid Network',
        'Custody Model': 'Federated sidechain, 11-of-15 functionaries',
        'Trusted Parties': 15,
        'Exit Mechanism': 'Federation-controlled peg-out only',
        'Time to Unilateral Exit': 'No unilateral exit',
        'Online Requirement': 'Sender only at send time',
        'Transfer Speed': '~1 minute',
        'Typical Fee Range': '<$0.10',
    },
    {
        'Layer': 'Custodial (exchange)',
        'Custody Model': 'Full custody by third party',
        'Trusted Parties': 1,  # the custodian
        'Exit Mechanism': 'Withdrawal request, subject to custodian policy',
        'Time to Unilateral Exit': 'No unilateral exit',
        'Online Requirement': 'Custodian operational',
        'Transfer Speed': 'Instant internal',
        'Typical Fee Range': '$0 internal, withdrawal fees apply',
    },
])

trust_matrix

## 7. Cost Comparison Across Transfer Sizes

For each layer, compute the fee in basis points (bps) for moving $100, $1,000, and $10,000 worth of value. Lower bps means cheaper relative cost. Where on-chain Bitcoin fees are flat per transaction, the bps cost decreases as transfer size grows. Off-chain layers stay roughly flat regardless of size.

In [ ]:
# Approximate flat fees in USD per transaction
# Update L1 fee estimate based on current mempool conditions
l1_flat_fee_usd = 2.50    # rough median, varies wildly with mempool
ln_flat_fee_usd = 0.005   # typical small Lightning fee
spark_flat_fee_usd = 0.001  # essentially free on-network
liquid_flat_fee_usd = 0.05  # typical Liquid tx fee
custodial_internal_fee = 0  # internal transfers free

sizes = [100, 1000, 10000]
fee_table = []
for size in sizes:
    fee_table.append({
        'Transfer Size': f'${size:,}',
        'L1 Bitcoin (bps)': round(l1_flat_fee_usd / size * 10000, 2),
        'Lightning (bps)': round(ln_flat_fee_usd / size * 10000, 4),
        'Spark (bps)': round(spark_flat_fee_usd / size * 10000, 4),
        'Liquid (bps)': round(liquid_flat_fee_usd / size * 10000, 3),
        'Custodial Internal (bps)': 0,
    })

cost_df = pd.DataFrame(fee_table)
print("Cost in basis points (1 bps = 0.01%) by transfer size:")
print()
cost_df

## 8. Chart One: Lightning Capacity vs BTC Price

Does Lightning Network capacity grow with BTC price (speculation following), against price drops (defensive HODLer behavior), or independently (utility-driven)? The relationship reveals whether Lightning is being used as payment rail or treated as a price-correlated speculative bet.

In [ ]:
if ln_df is not None and btc_df is not None:
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Lightning capacity on left axis
    ax1.plot(ln_df['date'], ln_df['capacity_btc'], color='#f7931a', linewidth=2, label='LN Public Capacity (BTC)')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Lightning Capacity (BTC)', color='#f7931a')
    ax1.tick_params(axis='y', labelcolor='#f7931a')

    # BTC price on right axis
    ax2 = ax1.twinx()
    ax2.plot(btc_df['date'], btc_df['price'], color='#2c3e50', linewidth=1.5, alpha=0.7, label='BTC/USD')
    ax2.set_ylabel('BTC Price (USD)', color='#2c3e50')
    ax2.tick_params(axis='y', labelcolor='#2c3e50')

    plt.title('Lightning Network Capacity vs BTC Price (Last 12 Months)', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.savefig('chart_ln_capacity_vs_price.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Chart saved to chart_ln_capacity_vs_price.png")
else:
    print("Cannot generate chart, missing data")

## 9. Chart Two: Bitcoin Locked Across L2s

Horizontal bar chart comparing BTC committed to each Layer 2 rail. This is the visual punchline: even years after launch, the absolute amount of BTC trusting non-base-chain rails is small relative to circulating supply (~19.7M BTC).

In [ ]:
# Approximate circulating supply
btc_circulating = 19_750_000

l2_data = pd.DataFrame([
    {'Layer': 'Lightning (public)', 'BTC Locked': ln_capacity_btc},
    {'Layer': 'Liquid', 'BTC Locked': liquid_btc},
    {'Layer': 'Spark (estimated)', 'BTC Locked': spark_btc_estimated},
])
l2_data['Pct of Circulating'] = (l2_data['BTC Locked'] / btc_circulating) * 100
l2_data = l2_data.sort_values('BTC Locked', ascending=True)

fig, ax = plt.subplots(figsize=(11, 5))
colors = ['#9b59b6', '#3498db', '#f7931a']
bars = ax.barh(l2_data['Layer'], l2_data['BTC Locked'], color=colors)

for bar, btc, pct in zip(bars, l2_data['BTC Locked'], l2_data['Pct of Circulating']):
    width = bar.get_width()
    ax.text(width + max(l2_data['BTC Locked']) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{btc:,.0f} BTC ({pct:.4f}%)',
            ha='left', va='center', fontsize=10)

ax.set_xlabel('BTC Locked')
ax.set_title('Bitcoin Committed to L2 Rails (with % of Circulating Supply)',
             fontsize=14, fontweight='bold')
ax.set_xlim(0, max(l2_data['BTC Locked']) * 1.3)
plt.tight_layout()
plt.savefig('chart_btc_in_l2s.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to chart_btc_in_l2s.png")
print()
print(l2_data.to_string(index=False))

## 10. Takeaways for the Written Report

Use these as discussion prompts when drafting your Substack piece:

1. **Trust is multi-dimensional.** Every L2 makes a different bet (channel partners, federations, threshold operators, custodians). Reducing this to "trustless vs trusted" loses the actual decision being made.

2. **Unilateral exit is the strongest property.** L1 and Lightning have it. Spark has a slower version. Liquid and custodial do not. This is the highest-stakes column in the matrix.

3. **Cost-per-byte advantage flips with size.** L1 is too expensive for $5 coffees but cheapest as a percentage for $1M settlement. Off-chain rails dominate small transfers but should not be the destination for life savings.

4. **Adoption tells a story about preference revelation.** Lightning capacity is X BTC. Liquid is Y BTC. Spark is Z BTC. Compare against the ~19.7M BTC outstanding. The absolute numbers are small. What does that say about user preferences for self-custody on L1 versus convenience on L2?

5. **Spark's bet is fixing Lightning's UX problems.** No channel management, offline receiving, native tokens. The tradeoff is the operator trust assumption. Whether users accept that tradeoff is the open question.

---

**Next steps:**

- Save final values from the notebook output
- Write the report (800 to 1,200 words) using these data points
- Push notebook + charts to GitHub  
- Post on Substack and X with the chart embedded